# 01. Calibration NPZ

Detects the ArUco marker in one frame of each calibration video and saves the camera extrinsics (R, t).
Run this only after moving a camera; otherwise reuse the existing NPZ files. Existing NPZ files are replaced only if `OVERWRITE` is checked.

| | |
|---|---|
| Input | `calibration.top_video`, `calibration.front_video` in `setup/` (frame `calibration.frame_index`) |
| Output | `inputs.calib_top_npz`, `inputs.calib_front_npz`, and a check image next to each NPZ (`<NPZ name>_check.png`) |
| GPU | Not required |

Cell 0 asks for the day folder (`Top/`, `Front/`, `setup/`, `result/`). The last folder is remembered; press Enter to reuse it.
If `setup/experiment_settings.yaml` does not exist, a form opens: fill it in and click **Save**. Otherwise the current settings are shown with **Edit** buttons.

In [ ]:
#@title 0. Setup: repository, Drive, day folder, day settings
REPO_URL = "https://github.com/dodokim777-boop/underwater-gripper-tracking.git"  #@param {type:"string"}

import os, sys, subprocess, warnings
REPO_DIR = '/content/' + REPO_URL.rstrip('/').split('/')[-1].removesuffix('.git')
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=False)
%pip install -q -r {REPO_DIR}/requirements.txt

from google.colab import drive
drive.mount('/content/drive')

sys.path.insert(0, REPO_DIR)
import pandas as pd
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
from main_code import settings
from main_code.common.notebook_ui import ask_day_folder, ask_trial, day_settings

DAY = ask_day_folder()
SETTINGS_FILE = day_settings(DAY, os.path.join(REPO_DIR, 'experiment_settings.yaml'))

In [ ]:
#@title 1. Load and check settings
settings.configure(settings.day_settings_path(DAY), data_root=DAY)
settings.print_summary('calibration')
settings.check('calibration')

## 2. Detect markers and save

Check the images: X (red) to the right, Y (green) into the tank, Z (blue) up.
If a marker is not detected, set `calibration.frame_index` (cell 0, **Edit full file**) to a frame where the marker is clearly visible.

In [ ]:
#@title 2. Detect markers and save NPZ
OVERWRITE = False  #@param {type:"boolean"}

import cv2, matplotlib.pyplot as plt
from preparation.make_calibration_npz import make_calibration_npz

results = {role: make_calibration_npz(role, overwrite=OVERWRITE) for role in ('top', 'front')}

fig, axes = plt.subplots(1, 2, figsize=(18, 5.5))
for ax, (role, r) in zip(axes, results.items()):
    ax.imshow(cv2.cvtColor(cv2.imread(r['check_image']), cv2.COLOR_BGR2RGB))
    ax.set_title(role); ax.axis('off')
plt.tight_layout(); plt.show()